In [27]:
import json
import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd

# 1. Configurazione dinamica dei percorsi (Colab / IDE)
in_colab = "google.colab" in sys.modules

if in_colab:
    print("[INFO] Ambiente: Google Colab. Configurazione file nella root.")
    base_dir = "/content"
    # Se su Colab carica i file caricati in /content
    tracking_parquet_path = os.path.join(base_dir, "integrated_tracking_data.parquet")
    opta_match_json_path = os.path.join(base_dir, "MA1_opta_match.json")
    opta_events_json_path = os.path.join(base_dir, "MA3_opta_matchevent.json")
else:
    print("[INFO] Ambiente: Local IDE (IntelliJ). Configurazione radice locale.")
    current_dir = (
        os.path.dirname(os.path.abspath(__file__))
        if "__file__" in locals()
        else os.getcwd()
    )
    base_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))

    # Su IntelliJ mantiene l'architettura data/raw e data/processed
    tracking_parquet_path = os.path.join(base_dir, "data", "processed", "integrated_tracking_data.parquet")
    opta_match_json_path = os.path.join(base_dir, "data", "raw", "MA1_opta_match.json")
    opta_events_json_path = os.path.join(base_dir, "data", "raw", "MA3_opta_matchevent.json")



# 2. Caricamento e normalizzazione cronologica degli eventi OPTA
print("\n[FASE 1] Parsing e allineamento temporale del flusso eventi JSON...")
with open(opta_events_json_path, "r", encoding="utf-8") as f:
    eventi_data = json.load(f)

events_list = eventi_data["liveData"]["event"]

parsed_events = []
for ev in events_list:
    dt_str = ev["timeStamp"].replace("Z", "")
    dt_obj = datetime.fromisoformat(dt_str)
    ts_ms = int(dt_obj.timestamp() * 1000)

    parsed_events.append(
        {
            "event_id": ev["id"],
            "type_id": ev["typeId"],
            "period_id": ev["periodId"],
            "min": ev["timeMin"],
            "sec": ev["timeSec"],
            "team_id": ev.get("contestantId"),
            "outcome": ev.get("outcome", 1),
            "timestamp_ms": ts_ms,
        }
    )

df_events = (
    pd.DataFrame(parsed_events).sort_values("timestamp_ms").reset_index(drop=True)
)
print(f"-> Mappati con successo {len(df_events)} eventi Opta Vision.")




# 3. Segmentazione temporale e normalizzazione spaziale del tracking
print("\n[FASE 2] Caricamento del dataset posizionale ad alta frequenza (Parquet)...")

t1_start = df_events[df_events["period_id"] == 1]["timestamp_ms"].min()
t1_end = df_events[df_events["period_id"] == 1]["timestamp_ms"].max()
t2_start = df_events[df_events["period_id"] == 2]["timestamp_ms"].min()
t2_end = df_events[df_events["period_id"] == 2]["timestamp_ms"].max()

df_tracking = pd.read_parquet(tracking_parquet_path)

df_tracking["periodo"] = 0
df_tracking.loc[
    (df_tracking["timestamp"] >= t1_start) & (df_tracking["timestamp"] <= t1_end),
    "periodo",
] = 1
df_tracking.loc[
    (df_tracking["timestamp"] >= t2_start) & (df_tracking["timestamp"] <= t2_end),
    "periodo",
] = 2

df_tracking = df_tracking[df_tracking["periodo"].isin([1, 2])].copy()

df_tracking.loc[df_tracking["periodo"] == 2, "x"] = -df_tracking.loc[
    df_tracking["periodo"] == 2, "x"
]
df_tracking.loc[df_tracking["periodo"] == 2, 'y'] = -df_tracking.loc[df_tracking['periodo'] == 2, 'y']
print(f"-> Inversione di campo applicata su {len(df_tracking)} righe di tracciamento.")





# 4. Identificazione dei ruoli ed ingegnerizzazione delle feature relative
print("\n[FASE 3] Isolamento dei portieri ed estrazione delle metriche relative...")

id_portieri = (
    df_tracking.groupby(["team", "player_id"])["x"].mean().reset_index()
)
id_gk_H = (
    id_portieri[id_portieri["team"] == "H"]
    .sort_values(by="x")
    .iloc[0]["player_id"]
)
id_gk_A = (
    id_portieri[id_portieri["team"] == "A"]
    .sort_values(by="x", ascending=False)
    .iloc[0]["player_id"]
)

print(f"-> Portiere Squadra di Casa (H) rilevato: {id_gk_H}")
print(f"-> Portiere Squadra Ospite (A) rilevato: {id_gk_A}")

# Si considera come nei test il campo lungo 105 e largo 68 con origine (0;0) situato a cenrocampo
def prepara_dati_scientifici(df_track):
    X_MIN, X_MAX = -52.5, 52.5
    Y_MIN, Y_MAX = -34.0, 34.0
    PITCH_LENGTH, PITCH_WIDTH = 105.0, 68.0

    df_clean = df_track.dropna(subset=["x", "y"]).copy()

    x_clamped = np.clip(df_clean["x"], X_MIN, X_MAX)
    y_clamped = np.clip(df_clean["y"], Y_MIN, Y_MAX)

    df_clean["x_norm"] = (
        (x_clamped - X_MIN) / (X_MAX - X_MIN)
    ) * PITCH_LENGTH
    df_clean["y_norm"] = (
        (y_clamped - Y_MIN) / (Y_MAX - Y_MIN)
    ) * PITCH_WIDTH

    is_not_gk = (df_clean["player_id"] != id_gk_H) & (
        df_clean["player_id"] != id_gk_A
    )

    centroide_frame = (
        df_clean[is_not_gk]
        .groupby(["timestamp", "team"])[["x_norm", "y_norm"]]
        .mean()
        .reset_index()
    )
    centroide_frame.rename(columns={"x_norm": "cx", "y_norm": "cy"}, inplace=True)

    df_clean = pd.merge(
        df_clean, centroide_frame, on=["timestamp", "team"], how="left"
    )

    df_clean["x_rel"] = df_clean["x_norm"] - df_clean["cx"]
    df_clean["y_rel"] = df_clean["y_norm"] - df_clean["cy"]

    df_clean = df_clean.sort_values(by=["player_id", "timestamp"])
    df_clean["vx"] = df_clean.groupby("player_id")["x_norm"].diff()
    df_clean["vy"] = df_clean.groupby("player_id")["y_norm"].diff()
    df_clean["velocita"] = np.sqrt(df_clean["vx"] ** 2 + df_clean["vy"] ** 2)

    return df_clean.dropna(subset=["x_rel", "y_rel"])


df_tattico = prepara_dati_scientifici(df_tracking)

df_tattico = df_tattico[
    (df_tattico["player_id"] != id_gk_H) & (df_tattico["player_id"] != id_gk_A)
].copy()

print(
    f"\n=== PIPELINE DI PRE-PROCESSING COMPLETATA: {len(df_tattico)} RECORD PRONTI PER I MODELLI TATTICI ==="
)

[INFO] Ambiente: Local IDE (IntelliJ). Configurazione radice locale.

[FASE 1] Parsing e allineamento temporale del flusso eventi JSON...
-> Mappati con successo 1688 eventi Opta Vision.

[FASE 2] Caricamento del dataset posizionale ad alta frequenza (Parquet)...
-> Inversione di campo applicata su 2493702 righe di tracciamento.

[FASE 3] Isolamento dei portieri ed estrazione delle metriche relative...
-> Portiere Squadra di Casa (H) rilevato: d2hnxi1yi5rgqj1tvxh1813x0
-> Portiere Squadra Ospite (A) rilevato: ujb2hkujcnwnyrbsgk3ilmtx

=== PIPELINE DI PRE-PROCESSING COMPLETATA: 2243511 RECORD PRONTI PER I MODELLI TATTICI ===


In [28]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score

from utils_graphics import visualizza_lavagnetta_ungherese

# 1. Algoritmo di clustering (MODELLO A)
def individua_modulo_simmetrico(df_finestra_squadra, team_code=None):
    """
    Rileva il modulo tattico applicando una logica basata su distance_threshold=7.5.
    Mantiene intatta l'asimmetria geometrica dei reparti scaglionati del calcio fluido.
    I dati di tracking condividono la medesima direzione di attacco nativa nel dataset.
    """
    X = df_finestra_squadra[["x_rel", "y_rel"]].values
    if len(X) < 110:
        return "N/A", None, None

    # Identificazione dei 10 centroidi stabili spaziali
    kmeans_ruoli = KMeans(n_clusters=10, random_state=42, n_init=10)
    kmeans_ruoli.fit(X)
    centri_ruoli_ideali = kmeans_ruoli.cluster_centers_

    # Ordinamento dei 10 ruoli stabili lungo la profondità del campo (asse X)
    ordine_x_iniziale = np.argsort(centri_ruoli_ideali[:, 0])
    centri_ruoli_ideali = centri_ruoli_ideali[ordine_x_iniziale]

    altezze_x = centri_ruoli_ideali[:, 0].reshape(-1, 1)

    # Clustering Gerarchico sulla profondità con la soglia (7.5 metri)
    # Questa distanza ristretta preserva le asimmetrie reali e lo scaglionamento dei reparti
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=7.5,
        metric="euclidean",
        linkage="complete",
    )
    clustering.fit(altezze_x)
    labels_grezze = clustering.labels_
    num_reparti = clustering.n_clusters_

    # Fallback controllato su 3 o 4 linee (autonomo) basato su densità (Silhouette Score)
    if num_reparti < 3 or num_reparti > 4:
        km3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(altezze_x)
        km4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit(altezze_x)
        if silhouette_score(altezze_x, km3.labels_) >= silhouette_score(
            altezze_x, km4.labels_
        ):
            labels_grezze = km3.labels_
            num_reparti = 3
        else:
            labels_grezze = km4.labels_
            num_reparti = 4

    # Ordinamento coerente dei reparti (0=Difesa, 1=Centrocampo, ecc.)
    centri_cluster = []
    for r in range(num_reparti):
        centri_cluster.append(np.mean(altezze_x[labels_grezze == r]))

    ordine_reparti = np.argsort(centri_cluster)
    mappa_ordine = {
        vecchio: nuovo for nuovo, vecchio in enumerate(ordine_reparti)
    }
    labels_reparti = np.array([mappa_ordine[l] for l in labels_grezze])

    # Costruzione della stringa del modulo (Garantisce la lettura corretta: D-C-A)
    componenti = [
        str(np.sum(labels_reparti == r)) for r in range(num_reparti)
    ]
    str_modulo = "-".join(componenti)

    return str_modulo, centri_ruoli_ideali, labels_reparti


# 2. Pipeline di stampa congiunta
# Suddivisione discreta dei timestamp in 6 intervalli temporali omogenei da 15 minuti
unique_timestamps = sorted(df_tattico["timestamp"].unique())
chunks = np.array_split(unique_timestamps, 6)

print(
    "=== AVVIO ESTRAZIONE GRAFICA CONGIUNTA: PIPELINE CON STRUTTURA FLUIDA ==="
)

for i, chunk in enumerate(chunks, start=1):
    df_chunk = df_tattico[df_tattico["timestamp"].isin(chunk)]
    testo_fase = (
        f"Fase di gioco {i} (Minuti indicativi: {int((i - 1) * 15)}' - {int(i * 15)}')"
    )

    print(f"\n" + "=" * 72)
    print(f"[ELABORAZIONE CHUNK FLUIDO] {testo_fase}")
    print("=" * 72)

    # Iterazione sequenziale sulle due squadre nello stesso blocco temporale
    for team_code in ["H", "A"]:
        nome_team = "H (Casa)" if team_code == "H" else "A (Ospite)"

        # Isolamento del subset posizionale specifico per la squadra corrente
        df_team_chunk = df_chunk[df_chunk["team"] == team_code]

        # Computazione dell'algoritmo fluido asimmetrico originale
        modulo, centri, labels = individua_modulo_simmetrico(
            df_team_chunk, team_code
        )

        # Controllo di consistenza sul volume minimo dei dati disponibili nel chunk
        if modulo == "N/A" or centri is None or labels is None:
            print(
                f"       [AVVISO] Dati insufficienti per {nome_team} nella Fase {i}. Salto il grafico."
            )
            continue

        print(f"       -> {nome_team} | Modulo Rilevato: {modulo}")

        # Passa esplicitamente il nome della cartella di destinazione
        # Le stampe vengono salvate in out/geometricAnalysis/tactics01
        visualizza_lavagnetta_ungherese(
            nome_team, team_code, testo_fase, modulo, centri, labels, nome_script="tactics01"
        )

print("\n=== PIPELINE DOPPIA COMPLETATA CON SUCCESSO ===")

=== AVVIO ESTRAZIONE GRAFICA CONGIUNTA: PIPELINE CON STRUTTURA FLUIDA ===

[ELABORAZIONE CHUNK FLUIDO] Fase di gioco 1 (Minuti indicativi: 0' - 15')
       -> H (Casa) | Modulo Rilevato: 2-4-4
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\tactics01\H_(Casa)_Fase_di_gioco_1.png
       -> A (Ospite) | Modulo Rilevato: 2-2-3-3
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\tactics01\A_(Ospite)_Fase_di_gioco_1.png

[ELABORAZIONE CHUNK FLUIDO] Fase di gioco 2 (Minuti indicativi: 15' - 30')
       -> H (Casa) | Modulo Rilevato: 1-4-3-2
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\tactics01\H_(Casa)_Fase_di_gioco_2.png
       -> A (Ospite) | Modulo Rilevato: 3-3-3-1
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C

In [29]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import os
import sys
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.metrics import silhouette_score

# Importazione del modulo delle grafiche
from utils_graphics import visualizza_lavagnetta_ungherese

# 1. Algoritmo di clustering (MODELLO B)
# ==============================================================================
def individua_modulo_simmetrico_avanzato(df_finestra_squadra, team_code=None):
    """
    Rileva il modulo tattico forzando la stabilità simmetrica dei reparti (tolleranza 9.5m).
    Assorbe le fluttuazioni asimmetriche laterali all'interno delle macro-linee di gioco.
    Tratta in modo omogeneo Casa e Ospite sulla base della direzione di attacco nativa condivisa.
    """
    X = df_finestra_squadra[["x_rel", "y_rel"]].values
    if len(X) < 110:
        return "N/A", None, None

    # Identificazione dei 10 centroidi stabili spaziali
    kmeans_ruoli = KMeans(n_clusters=10, random_state=42, n_init=10)
    kmeans_ruoli.fit(X)
    centri_ruoli_ideali = kmeans_ruoli.cluster_centers_

    # Ordinamento dei 10 ruoli stabili lungo la profondità del campo (asse X delle coordinate relative)
    idx_ordine_tattico = np.argsort(centri_ruoli_ideali[:, 0])
    centri_ruoli_ideali = centri_ruoli_ideali[idx_ordine_tattico]

    altezze_x = centri_ruoli_ideali[:, 0].reshape(-1, 1)

    # Clustering Gerarchico sulla profondità con soglia allargata a 9.5 metri
    # L'innalzamento della tolleranza a 9.5m è finalizzato al riallineamento geometrico dei ruoli scaglionati
    clustering = AgglomerativeClustering(
        n_clusters=None,
        distance_threshold=9.5,
        metric="euclidean",
        linkage="complete",
    )
    clustering.fit(altezze_x)
    labels_grezze = clustering.labels_
    num_reparti = clustering.n_clusters_

    # Fallback controllato su 3 o 4 linee (autonomo) basato su densità (Silhouette Score)
    if num_reparti < 3 or num_reparti > 4:
        km3 = KMeans(n_clusters=3, random_state=42, n_init=10).fit(altezze_x)
        km4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit(altezze_x)
        if silhouette_score(altezze_x, km3.labels_) >= silhouette_score(
                altezze_x, km4.labels_
        ):
            labels_grezze = km3.labels_
            num_reparti = 3
        else:
            labels_grezze = km4.labels_
            num_reparti = 4

    # Ordinamento dei reparti dal settore più arretrato a quello più avanzato
    centri_cluster = []
    for r in range(num_reparti):
        centri_cluster.append(np.mean(altezze_x[labels_grezze == r]))

    ordine_reparti = np.argsort(centri_cluster)
    mappa_ordine = {
        vecchio: nuovo for nuovo, vecchio in enumerate(ordine_reparti)
    }
    labels_reparti = np.array([mappa_ordine[l] for l in labels_grezze])

    # Generazione della stringa normalizzata del modulo (Difesa-Centrocampo-Attacco)
    componenti = [
        str(np.sum(labels_reparti == r)) for r in range(num_reparti)
    ]
    str_modulo = "-".join(componenti)

    return str_modulo, centri_ruoli_ideali, labels_reparti


# 2. Pipeline di stampa congiunta
# Suddivisione discreta dei timestamp in 6 intervalli temporali omogenei da 15 minuti
unique_timestamps = sorted(df_tattico["timestamp"].unique())
chunks = np.array_split(unique_timestamps, 6)

print("=== AVVIO ESTRAZIONE GRAFICA: PIPELINE RIGIDA SIMMETRICA ===")

for i, chunk in enumerate(chunks, start=1):
    df_chunk = df_tattico[df_tattico["timestamp"].isin(chunk)]
    testo_fase = (
        f"Fase di gioco {i} (Minuti indicativi: {int((i-1)*15)}' - {int(i*15)}')"
    )

    print(f"\n" + "=" * 72)
    print(f"[ELABORAZIONE CHUNK MODELLO SIMMETRICO] {testo_fase}")
    print("=" * 72)

    # Iterazione sequenziale sui due club per l'estrazione simultanea delle lavagnette
    for team_code in ["H", "A"]:
        nome_team = "H (Casa)" if team_code == "H" else "A (Ospite)"
        df_team_chunk = df_chunk[df_chunk["team"] == team_code]

        # Computazione dell'algoritmo a tolleranza macro-strutturale
        modulo, centri, labels = individua_modulo_simmetrico_avanzato(
            df_team_chunk, team_code
        )

        # Verifica di consistenza volumetrica dei record nel chunk
        if modulo == "N/A" or centri is None or labels is None:
            print(
                f"       [AVVISO] Dati insufficienti per {nome_team} nella Fase {i}. Salto il grafico."
            )
            continue

        print(f"       -> {nome_team} | Modulo Rilevato: {modulo}")

        # Passa esplicitamente il nome della cartella di destinazione
        # Le stampe vengono salvate in out/geometricAnalysis/tactics02
        visualizza_lavagnetta_ungherese(
            nome_team, team_code, testo_fase, modulo, centri, labels, nome_script="tactics02"
        )

print("\n=== PIPELINE DOPPIA COMPLETATA ===")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
=== AVVIO ESTRAZIONE GRAFICA: PIPELINE RIGIDA SIMMETRICA ===

[ELABORAZIONE CHUNK MODELLO SIMMETRICO] Fase di gioco 1 (Minuti indicativi: 0' - 15')
       -> H (Casa) | Modulo Rilevato: 3-3-4
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\tactics02\H_(Casa)_Fase_di_gioco_1.png
       -> A (Ospite) | Modulo Rilevato: 4-3-3
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\tactics02\A_(Ospite)_Fase_di_gioco_1.png

[ELABORAZIONE CHUNK MODELLO SIMMETRICO] Fase di gioco 2 (Minuti indicativi: 15' - 30')
       -> H (Casa) | Modulo Rilevato: 1-4-3-2
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\tactics02\H_(Casa)_Fase_di_gioco_2.png
       -> 

In [30]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import json
import os
import sys
from datetime import datetime
import numpy as np
import pandas as pd

# Importazione del modulo delle grafiche
from utils_graphics import visualizza_lavagnetta_ungherese


# 1. Configurazione dinamica dei percorsi (Colab / IDE)
in_colab = "google.colab" in sys.modules

if in_colab:
    print("[INFO] Ambiente: Google Colab. Configurazione file eventi nella root.")
    base_dir = "/content"
    opta_events_json_path = os.path.join(base_dir, "MA3_opta_matchevent.json")
else:
    print("[INFO] Ambiente: Local IDE (IntelliJ). Configurazione radice locale.")
    current_dir = (
        os.path.dirname(os.path.abspath(__file__))
        if "__file__" in locals()
        else os.getcwd()
    )
    # Sale di due livelli per puntare alla radice del progetto e poi cerca la cartella così come stabilito
    base_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))
    opta_events_json_path = os.path.join(
        base_dir, "data", "raw", "MA3_opta_matchevent.json"
    )


# 2. Rricostruzione della timeline del possesso
print("\n[FASE 1] Analisi del flusso eventi JSON per ricostruzione timeline possesso...")
with open(opta_events_json_path, "r", encoding="utf-8") as f:
    eventi_json = json.load(f)

events = eventi_json["liveData"]["event"]

# Codice alfanumerico univoco associato al club di casa per la discriminazione del possesso
id_H = "3vo5mpj7catp66nrwwqiuhuup"

timeline_possesso = []

for i in range(len(events) - 1):
    ev_attuale = events[i]
    ev_successivo = events[i + 1]

    ts_attuale = int(
        datetime.fromisoformat(
            ev_attuale["timeStamp"].replace("Z", "")
        ).timestamp()
        * 1000
    )
    ts_successivo = int(
        datetime.fromisoformat(
            ev_successivo["timeStamp"].replace("Z", "")
        ).timestamp()
        * 1000
    )

    if ev_attuale.get("typeId") == 1 and ev_attuale.get("outcome") == 1:
        squadra_in_possesso = (
            "H" if ev_attuale.get("contestantId") == id_H else "A"
        )
        palla_in_gioco = True
    elif ev_attuale.get("typeId") in [4, 5, 6, 8, 30]:  # ID standard interruzioni Opta
        palla_in_gioco = False
        squadra_in_possesso = "Nessuno"
    else:
        continue

    timeline_possesso.append(
        {
            "ts_start": ts_attuale,
            "ts_end": ts_successivo,
            "stato_possesso": squadra_in_possesso,
            "palla_attiva": palla_in_gioco,
        }
    )

df_timeline = pd.DataFrame(timeline_possesso)
print(f"-> Timeline del possesso ricostruita. Mappati {len(df_timeline)} segmenti fluidi.")


# 3. Sincronizzazione ad alta frequenza tra tracking ed eventi
print("\n[FASE 2] Sincronizzazione cinematica (Merge Asof ad alta frequenza)...")
df_tattico = df_tattico.sort_values("timestamp")
df_timeline = df_timeline.sort_values("ts_start")

df_tattico_filtrato = pd.merge_asof(
    df_tattico,
    df_timeline,
    left_on="timestamp",
    right_on="ts_start",
    direction="backward",
)

# Esclude i tempi morti tenendo solo l'Open Play attivo
df_tattico_filtrato = df_tattico_filtrato[
    df_tattico_filtrato["palla_attiva"] == True
    ].copy()

print(f"-> Sincronizzazione completata. Record residui in Open Play: {len(df_tattico_filtrato)}")


# 4. Pipeline di stampa binaria con modello B (Soglia bilanciata 9.5m)
unique_timestamps = sorted(df_tattico_filtrato["timestamp"].unique())
chunks = np.array_split(unique_timestamps, 6)

print("\n=== AVVIO ESTRAZIONE GRAFICA SCIENTIFICA DIVISA PER FASE - MODELLO B ===")

for i, chunk in enumerate(chunks, start=1):
    df_chunk = df_tattico_filtrato[df_tattico_filtrato["timestamp"].isin(chunk)]
    testo_fase = f"Fase di gioco {i} (Minuti indicativi: {int((i - 1) * 15)}' - {int(i * 15)}')"

    print(f"\n" + "=" * 72)
    print(f"[ELABORAZIONE CHUNK CONTESTUALE] {testo_fase}")
    print("=" * 72)

    for team_code in ["H", "A"]:
        nome_team = "H (Casa)" if team_code == "H" else "A (Ospite)"
        df_team_chunk = df_chunk[df_chunk["team"] == team_code]

        # Split tra fase difensiva e fase offensiva
        df_attacco = df_team_chunk[df_team_chunk["stato_possesso"] == team_code]
        df_difesa = df_team_chunk[
            (df_team_chunk["stato_possesso"] != team_code)
            & (df_team_chunk["stato_possesso"] != "Nessuno")
            ]

        # 1. Elaborazione Fase Offensiva (Modello B - 9.5m)
        modulo_att, centri_att, labels_att = individua_modulo_simmetrico_avanzato(df_attacco, team_code)
        if modulo_att != "N/A" and centri_att is not None:
            print(f"       -> {nome_team} in FASE DI POSSESSO | Modulo: {modulo_att}")
            # Salva i risultati in out/geometricAnalysis/possession01/
            visualizza_lavagnetta_ungherese(
                f"{nome_team} - Fase Offensiva",
                team_code,
                testo_fase,
                modulo_att,
                centri_att,
                labels_att,
                nome_script="possession01"
            )

        # 2. Elaborazione Fase Difensiva (Modello B - 9.5m)
        modulo_dif, centri_dif, labels_dif = individua_modulo_simmetrico_avanzato(df_difesa, team_code)
        if modulo_dif != "N/A" and centri_dif is not None:
            print(f"       -> {nome_team} in FASE DI NON POSSESSO | Modulo: {modulo_dif}")
            # Salva i risultati in out/geometricAnalysis/possession01/
            visualizza_lavagnetta_ungherese(
                f"{nome_team} - Fase Difensiva",
                team_code,
                testo_fase,
                modulo_dif,
                centri_dif,
                labels_dif,
                nome_script="possession01"
            )

print("\n=== ANALISI SCIENTIFICA CONTESTUALE COMPLETATA CON SUCCESSO ===")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
[INFO] Ambiente: Local IDE (IntelliJ). Configurazione radice locale.

[FASE 1] Analisi del flusso eventi JSON per ricostruzione timeline possesso...
-> Timeline del possesso ricostruita. Mappati 1123 segmenti fluidi.

[FASE 2] Sincronizzazione cinematica (Merge Asof ad alta frequenza)...
-> Sincronizzazione completata. Record residui in Open Play: 1440528

=== AVVIO ESTRAZIONE GRAFICA SCIENTIFICA DIVISA PER FASE - MODELLO B ===

[ELABORAZIONE CHUNK CONTESTUALE] Fase di gioco 1 (Minuti indicativi: 0' - 15')
       -> H (Casa) in FASE DI POSSESSO | Modulo: 4-2-4
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\football-formation-analytics\out\geometricAnalysis\possession01\H_(Casa)___Fase_Offensiva_Fase_di_gioco_1.png
       -> H (Casa) in FASE DI NON POSSESSO | Modulo: 5-3-2
       [GRAFICO OMNI] Lavagnetta salvata con successo in: C:\Users\fzamb\IdeaProjects\foot